* [#59](https://github.com/salgo60/spa2Commons/issues/59) / [#64](https://github.com/salgo60/spa2Commons/issues/64)
* [59_SPA_dubbletter_3](https://github.com/salgo60/spa2Commons/blob/main/Notebook/59_SPA_dubbletter_3.ipynb)

SPA dubbletter men hämta bilden via https://portrattarkiv.se/endpoints/file.php?id=sj9PGLAlnmUAAAAAACcZwQ

In [1]:
from datetime import datetime

start_time = datetime.now()
print("Last run: ", start_time)

Last run:  2026-03-14 12:37:47.678512


In [2]:
import csv 
date_str = datetime.now().strftime("%Y%m%d")
csv_filename = f"portrait_duplicates_{date_str}.csv"

csv_file = open(csv_filename, "w", newline="", encoding="utf-8")
csv_writer = csv.writer(csv_file)

csv_writer.writerow([
    "item1_label",
    "item1_id",
    "item1_api",
    "item1_image",
    "item2_label",
    "item2_id",
    "item2_api",
    "item2_image",
    "distance"
])


96

In [3]:
from concurrent.futures import ThreadPoolExecutor
import requests
from PIL import Image
from io import BytesIO
import numpy as np
from collections import defaultdict
from tqdm import tqdm
import time
from datetime import datetime

SPARQL_URL = "https://query.wikidata.org/sparql"

QUERY = """
SELECT ?item ?itemLabel ?SPAid ?SPApicUrl ?api
WHERE {
  {
    SELECT ?item (COUNT(?SPAid) AS ?count)
    WHERE {
      ?item wdt:P4819 ?SPAid .
    }
    GROUP BY ?item
    ORDER BY DESC(?count)
    LIMIT 10
  }

  ?item wdt:P4819 ?SPAid .
  
  BIND(URI(CONCAT("https://portrattarkiv.se/endpoints/portraits.php?id=", ?SPAid)) AS ?api)
  BIND(URI(CONCAT("https://portrattarkiv.se/endpoints/file.php?id=", ?SPAid)) AS ?SPApicUrl)

  SERVICE wikibase:label { bd:serviceParam wikibase:language "sv,en" }
}
"""


def run_query():
    r = requests.get(
        SPARQL_URL,
        params={"query": QUERY, "format": "json"},
        headers={"User-Agent": "portrait-dhash-research"}
    )
    r.raise_for_status()
    return r.json()["results"]["bindings"]


def dhash(image, hash_size=8):
    image = image.convert("L").resize((hash_size + 1, hash_size))
    pixels = np.array(image)
    diff = pixels[:, 1:] > pixels[:, :-1]
    return diff.flatten()


def download_image(url):
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        return Image.open(BytesIO(r.content))
    except:
        return None

def fetch_image(row):

    spa_id = row["SPAid"]["value"]
    label = row["itemLabel"]["value"]
    img_url = row["SPApicUrl"]["value"]
    api = row["api"]["value"]

    img = download_image(img_url)

    return {
        "spa_id": spa_id,
        "label": label,
        "img": img,
        "img_url": img_url,
        "api": api
    }

# bucket storage
buckets = defaultdict(list)

data = run_query()

html_rows = []

similar_count = 0
tested_count = 0
html_rows = []
duplicate_rows = [] 
start_time = time.time()

with ThreadPoolExecutor(max_workers=20) as executor:

    results = executor.map(fetch_image, data)

    for item in tqdm(results, total=len(data), desc="Processing portraits"):

        img = item["img"]

        if img is None:
            continue

        spa_id = item["spa_id"]
        label = item["label"]
        img_url = item["img_url"]
        api = item["api"]

        details_url = f"https://portrattarkiv.se/details/{spa_id}"

        tested_count += 1

        h = dhash(img)

        key = tuple(h[:16])

        similar_found = False
        matches = []

        for old_id, old_label, old_h, old_pic, old_api in buckets[key]:

            dist = np.sum(h != old_h)

            if dist <= 5:

                csv_writer.writerow([
                    label,
                    spa_id,
                    api,
                    img_url,
                    old_label,
                    old_id,
                    old_api,
                    old_pic,
                    dist
                ])
                similar_found = True
                similar_count += 1

                old_details = f"https://portrattarkiv.se/details/{old_id}"

                matches.append(f"""
                <div>
                <b>{old_label}</b><br>
                SPAID: {old_id}<br>
                distance: {dist}<br>

                <a href="{old_details}" target="_blank">details</a> |
                <a href="{old_api}" target="_blank">api</a> |
                <a href="{old_pic}" target="_blank">image</a>

                <br>

                <img src="{old_pic}" width="80">

                </div>
                """)

        bgcolor = "#ffcccc" if similar_found else ""

        row_html = f"""
        <tr style="background:{bgcolor}">
        
        <td>{tested_count}</td>
        
        <td>
        <a href="{details_url}" target="_blank">{label}</a><br>
        SPAID: {spa_id}<br>
        <a href="{api}" target="_blank">API</a>
        </td>
        
        <td>
        <img src="{img_url}" width="90">
        </td>
        
        <td>
        {''.join(matches)}
        </td>
        
        </tr>
        """
        
        html_rows.append(row_html)
        
        if similar_found:
            duplicate_rows.append(row_html)        

        buckets[key].append((spa_id, label, h, img_url, api))
    bgcolor = "#ffcccc" if similar_found else ""

    html_rows.append(f"""
    <tr style="background:{bgcolor}">
    
    <td>{tested_count}</td>

    <td>
    <a href="{details_url}" target="_blank">{label}</a><br>
    SPAID: {spa_id}<br>
    <a href="{api}" target="_blank">API</a>
    </td>

    <td>
    <a href="{img_url}" target="_blank">
    <img src="{img_url}" width="90">
    </a>
    </td>

    <td>
    {''.join(matches)}
    </td>

    </tr>
    """)

    buckets[key].append((spa_id, label, h, img_url, api))


# runtime statistics
end_time = time.time()
runtime = end_time - start_time

per_image = runtime / tested_count if tested_count else 0
estimate_900k = per_image * 900000


date_str = datetime.now().strftime("%Y%m%d")
filename = f"portrait_similarity_report_{date_str}.html"


html = f"""
<html>
<head>

<meta charset="utf-8">
<title>Portrait similarity report</title>

<style>

body {{
font-family: Arial;
}}

table {{
border-collapse: collapse;
}}

td,th {{
border:1px solid #ccc;
padding:5px;
}}

img {{
border-radius:4px;
}}

</style>

</head>

<body>

<h2>Portrait similarity report</h2>

<p>
Tested images: {tested_count}<br>
Similar matches: {similar_count}<br>
Runtime: {runtime:.2f} seconds<br>
Time per image: {per_image:.4f} sec<br>
Estimated runtime for 900000 images: {estimate_900k/3600:.2f} hours
</p>

<table>

<tr>
<th>#</th>
<th>Portrait</th>
<th>Image</th>
<th>Similar matches</th>
</tr>

{''.join(html_rows)}

</table>

</body>
</html>
"""

duplicates_filename = f"portrait_duplicates_report_{date_str}.html"

html_duplicates = f"""
<html>
<head>

<meta charset="utf-8">
<title>Portrait duplicates report</title>

<style>

body {{font-family: Arial}}

table {{border-collapse: collapse}}

td,th {{
border:1px solid #ccc;
padding:5px
}}

img {{border-radius:4px}}

</style>

</head>

<body>

<h2>Portrait duplicate report</h2>

<p>
Total tested: {tested_count}<br>
Duplicate matches: {similar_count}
</p>

<table>

<tr>
<th>#</th>
<th>Portrait</th>
<th>Image</th>
<th>Similar matches</th>
</tr>

{''.join(duplicate_rows)}

</table>

</body>
</html>
"""

with open(duplicates_filename, "w", encoding="utf-8") as f:
    f.write(html_duplicates)

print("Duplicate report written:", duplicates_filename)

with open(filename, "w", encoding="utf-8") as f:
    f.write(html)


print("HTML report written:", filename)
print("Tested:", tested_count)
print("Similar:", similar_count)
print("Runtime:", runtime)
print("Estimated runtime for 900k images (hours):", estimate_900k/3600)

Processing portraits: 100%|███████████████████| 343/343 [00:32<00:00, 10.47it/s]

Duplicate report written: portrait_duplicates_report_20260314.html
HTML report written: portrait_similarity_report_20260314.html
Tested: 343
Similar: 61
Runtime: 32.80349802970886
Estimated runtime for 900k images (hours): 23.909255123694503


In [4]:

csv_file.close()
print("CSV duplicate report written:", csv_filename)

CSV duplicate report written: portrait_duplicates_20260314.csv


In [8]:
from datetime import datetime

end = datetime.now()

print("Ended:", end)
print("Time elapsed:", end - start_time)

Ended: 2026-03-14 12:42:59.783651
Time elapsed: 0:00:00.936596
